# 不平衡处理

**负责人：** 6 号　**阶段：** 4 分析

比较三种策略，把最好的方案分享给全组。

## 任务
- [x] 不处理 / class_weight / 随机过采样（只在训练折内做）
- [x] 看 macro-F1 和各类别召回率
- [x] 每次实验先用 `new_run_id()` 生成 run_id

## 产出
- `results/imbalance.csv`
- `figures/analysis_imbalance.png`

使用 FastText mean 句向量 + Logistic Regression（不依赖 BiLSTM 或 Member 5）。

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if ROOT is None:
    raise RuntimeError("找不到仓库根目录")
sys.path.insert(0, str(ROOT))

from src.config import DATA_ROOT, FIGURES_DIR, LABELS, SEED, N_FOLDS
from src.data import load_clean, load_splits
from src.evaluate import evaluate, new_run_id, save_result
from src.cache import load_embedding, save_preds

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import RandomOverSampler

print("仓库根目录:", ROOT)

In [ ]:
clean = load_clean()
splits = load_splits(clean=clean)
label_of = dict(zip(clean["id"], clean["label"]))
dev = splits[splits["split"] == "train"].reset_index(drop=True)
test = splits[splits["split"] == "test"].reset_index(drop=True)
y_dev = dev["id"].map(label_of).to_numpy()
y_test = test["id"].map(label_of).to_numpy()

print(f"Train: {len(dev)}, Test: {len(test)}")
print("Class distribution (train):")
print(pd.Series(y_dev).map(LABELS).value_counts())

In [ ]:
def load_fold_vectors(fold):
    """Load FastText mean vectors for fold k (trained on the other 4 folds)."""
    train_ids = dev.loc[dev["fold"] != fold, "id"]
    val_ids = dev.loc[dev["fold"] == fold, "id"]
    X_train = load_embedding(f"fasttext_mean_fold{fold}", ids=train_ids)
    X_val = load_embedding(f"fasttext_mean_fold{fold}", ids=val_ids)
    y_train = y_dev[dev["fold"] != fold]
    y_val = y_dev[dev["fold"] == fold]
    return X_train, y_train, X_val, y_val

def load_full_vectors():
    """Full training + test vectors (trained on all training data)."""
    X_train = load_embedding("fasttext_mean_full", ids=dev["id"])
    X_test = load_embedding("fasttext_mean_full", ids=test["id"])
    return X_train, y_dev, X_test, y_test

In [ ]:
def run_strategy(strategy: str):
    """Run one imbalance strategy with 5-fold OOF + final test."""
    oof_proba = np.zeros((len(dev), 3))
    
    for k in range(N_FOLDS):
        X_tr, y_tr, X_va, y_va = load_fold_vectors(k)
        
        if strategy == "none":
            clf = LogisticRegression(max_iter=1000, random_state=SEED)
            clf.fit(X_tr, y_tr)
        elif strategy == "class_weight":
            classes = np.unique(y_tr)
            weights = compute_class_weight("balanced", classes=classes, y=y_tr)
            cw = dict(zip(classes, weights))
            clf = LogisticRegression(max_iter=1000, class_weight=cw, random_state=SEED)
            clf.fit(X_tr, y_tr)
        elif strategy == "oversample":
            ros = RandomOverSampler(random_state=SEED)
            X_res, y_res = ros.fit_resample(X_tr, y_tr)
            clf = LogisticRegression(max_iter=1000, random_state=SEED)
            clf.fit(X_res, y_res)
        else:
            raise ValueError(strategy)
        
        in_val = (dev["fold"] == k).to_numpy()
        oof_proba[in_val] = clf.predict_proba(X_va)
    
    # Final model on full training set
    X_full_tr, y_full_tr, X_te, y_te = load_full_vectors()
    if strategy == "none":
        final = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_full_tr, y_full_tr)
    elif strategy == "class_weight":
        classes = np.unique(y_full_tr)
        weights = compute_class_weight("balanced", classes=classes, y=y_full_tr)
        cw = dict(zip(classes, weights))
        final = LogisticRegression(max_iter=1000, class_weight=cw, random_state=SEED).fit(X_full_tr, y_full_tr)
    else:  # oversample
        ros = RandomOverSampler(random_state=SEED)
        X_res, y_res = ros.fit_resample(X_full_tr, y_full_tr)
        final = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_res, y_res)
    
    test_proba = final.predict_proba(X_te)
    return oof_proba, test_proba

In [ ]:
results = []
strategies = ["none", "class_weight", "oversample"]

for strat in strategies:
    print(f"Running strategy: {strat}")
    oof_proba, test_proba = run_strategy(strat)
    oof_pred = oof_proba.argmax(axis=1)
    
    run_id = new_run_id("imbalance", f"fasttext_mean_{strat}")
    row = evaluate(
        y_dev, oof_pred,
        run_id=run_id,
        model="imbalance",
        embedding="fasttext_mean",
        split="cv",
        params={"strategy": strat},
        notes=f"strategy={strat}"
    )
    save_result(row, "imbalance")
    
    # also save test metrics for reference
    test_pred = test_proba.argmax(axis=1)
    test_row = evaluate(
        y_test, test_pred,
        run_id=run_id,
        model="imbalance",
        embedding="fasttext_mean",
        split="test",
        params={"strategy": strat},
        notes=f"strategy={strat}"
    )
    save_result(test_row, "imbalance")
    
    save_preds(run_id, oof_ids=dev["id"].to_numpy(), oof_proba=oof_proba,
               test_ids=test["id"].to_numpy(), test_proba=test_proba, classes=[0,1,2])
    
    results.append({
        "strategy": strat,
        "macro_f1_cv": row["macro_f1"],
        "recall_neutral": row.get("recall_Neutral", row.get("recall_0")),
        "recall_positive": row.get("recall_Positive", row.get("recall_1")),
        "recall_negative": row.get("recall_Negative", row.get("recall_2")),
    })

df_res = pd.DataFrame(results)
print(df_res.to_string(index=False))
best = df_res.loc[df_res["macro_f1_cv"].idxmax(), "strategy"]
print(f"\nRecommended strategy: {best}")

In [ ]:
# Plot comparison
fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(strategies))
ax.bar(x, df_res["macro_f1_cv"], color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_xticks(x)
ax.set_xticklabels(strategies)
ax.set_ylabel("Macro-F1 (CV)")
ax.set_title("Imbalance Handling Comparison")
for i, v in enumerate(df_res["macro_f1_cv"]):
    ax.text(i, v + 0.005, f"{v:.3f}", ha="center")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "analysis_imbalance.png", dpi=200)
plt.show()